# Практическая работа №6: Подготовка фич и интеграция источников
**Цель работы:** Научиться объединять несколько источников данных (заказы, товары, события) и строить на их основе признаки (фичи) для последующей аналитики и простых ML‑моделей.

In [1]:
%pip install openpyxl

In [2]:
import glob
import os
import pandas as pd

data_dir = "data" if os.path.exists("data") else "."

# 1. Загружаем справочник товаров
df_products = pd.read_excel(os.path.join(data_dir, "products.xlsx"))

# 2. Загружаем и объединяем все файлы заказов
order_files = sorted(glob.glob(os.path.join(data_dir, "orders_2024_*.csv")))
df_orders = pd.concat([pd.read_csv(f) for f in order_files], ignore_index=True)

# Обогащаем заказы информацией о товарах и категориях
df_orders = df_orders.merge(
    df_products[["product_id", "category_id", "category_name"]],
    on="product_id",
    how="left",
)
df_orders["revenue"] = df_orders["price"] * df_orders["quantity"]

# 3. Находим для каждого пользователя его топ-категорию (top_category) по выручке
user_cat_revenue = (
    df_orders.groupby(["user_id", "category_name"])["revenue"]
    .sum()
    .reset_index()
)

user_top_category = (
    user_cat_revenue.sort_values(
        ["user_id", "revenue"], ascending=[True, False]
    )
    .groupby("user_id")
    .first()
    .reset_index()[["user_id", "category_name"]]
    .rename(columns={"category_name": "top_category"})
)

# 4. Загружаем события и объединяем с топ-категорией пользователя
df_events = pd.read_json(
    os.path.join(data_dir, "events_lines.json"), lines=True
)
df_events["event_time"] = pd.to_datetime(df_events["event_time"])
df_events = df_events.merge(user_top_category, on="user_id", how="left")

print("Топ-категории пользователей и логи событий подготовлены:")
print(user_top_category.head().to_string())

Топ-категории пользователей и логи событий подготовлены:
   user_id top_category
0     1001         Toys
1     1002         Toys
2     1003         Toys
3     1004        Sport
4     1005         Toys


In [3]:
# Флаг наличия события покупки (1 - purchase, 0 - любое другое событие)
df_events["is_purchase"] = (df_events["event_type"] == "purchase").astype(int)

# Перекрестная таблица (crosstab)
pivot_purchases = pd.crosstab(
    index=df_events["top_category"],
    columns=df_events["is_purchase"],
    rownames=["top_category"],
    colnames=["is_purchase (0=нет, 1=покупка)"],
)

print(
    "Перекрестная таблица событий по топ-категориям и совершению покупки:"
)
print(pivot_purchases.to_string())

Перекрестная таблица событий по топ-категориям и совершению покупки:
is_purchase (0=нет, 1=покупка)     0    1
top_category                             
Electronics                       50   11
Sport                            483  132
Toys                            1054  270


In [4]:
# Считаем общее число событий и долю покупок (конверсию) для каждой топ-категории
category_eval = pivot_purchases.copy()
category_eval["total_events"] = category_eval.sum(axis=1)
category_eval["purchase_rate"] = category_eval[1] / category_eval["total_events"]

# Сортируем по убыванию доли покупок
category_eval = category_eval.sort_values("purchase_rate", ascending=False)

print("Оценка категорий по конверсии в покупку:")
for cat, row in category_eval.iterrows():
  print(
      f"  - {cat}: совершено {int(row[1])} покупок из {int(row['total_events'])} событий "
      f"(доля: {row['purchase_rate'] * 100:.2f}%)"
  )

Оценка категорий по конверсии в покупку:
  - Sport: совершено 132 покупок из 615 событий (доля: 21.46%)
  - Toys: совершено 270 покупок из 1324 событий (доля: 20.39%)
  - Electronics: совершено 11 покупок из 61 событий (доля: 18.03%)


In [5]:
# 1. Формируем пользовательские сессии по логам событий
# (новая сессия начинается, если пауза между событиями пользователя превышает 30 минут)
df_events = df_events.sort_values(["user_id", "event_time"]).reset_index(
    drop=True
)
df_events["time_diff"] = df_events.groupby("user_id")["event_time"].diff()
df_events["is_new_session"] = (
    df_events["time_diff"] > pd.Timedelta(minutes=30)
).astype(int)
df_events["session_id"] = (
    df_events.groupby("user_id")["is_new_session"].cumsum()
)

# 2. Извлекаем просмотренный товар из URL страницы (/product/<id>)
df_events["page_product_id"] = (
    df_events["page"].str.extract(r"/product/(\d+)").astype(float)
)
df_events = df_events.merge(
    df_products[["product_id", "category_name"]],
    left_on="page_product_id",
    right_on="product_id",
    how="left",
)

# 3. Проверяем для каждого события: относится ли оно к топ-категории пользователя
df_events["is_in_top_category"] = (
    df_events["category_name"] == df_events["top_category"]
)

# 4. Агрегируем на уровне сессий:
# - была ли сессия в топ-категории
# - была ли в ней совершена покупка
session_agg = (
    df_events.groupby(["user_id", "session_id"])
    .agg(
        has_top_cat=("is_in_top_category", "max"),
        has_purchase=("is_purchase", "max"),
    )
    .reset_index()
)

# Оставляем только сессии, где пользователь взаимодействовал со своей топ-категорией
top_cat_sessions = session_agg[session_agg["has_top_cat"] == True]

# 5. Считаем долю сессий с покупкой в топ-категории (conversion_top_category)
user_conversion = (
    top_cat_sessions.groupby("user_id")["has_purchase"].mean().reset_index()
)
user_conversion.columns = ["user_id", "conversion_top_category"]

# 6. Объединяем признак с итоговым профилем пользователя
user_features = user_top_category.merge(user_conversion, on="user_id", how="left")
user_features["conversion_top_category"] = user_features[
    "conversion_top_category"
].fillna(0.0)

print(
    "Итоговый датафрейм пользователей с признаком conversion_top_category (первые 10 строк):"
)
print(user_features.head(10).to_string())

Итоговый датафрейм пользователей с признаком conversion_top_category (первые 10 строк):
   user_id top_category  conversion_top_category
0     1001         Toys                 0.000000
1     1002         Toys                 0.600000
2     1003         Toys                 0.444444
3     1004        Sport                 1.000000
4     1005         Toys                 0.500000
5     1006         Toys                 0.000000
6     1007         Toys                 0.000000
7     1008         Toys                 0.166667
8     1009        Sport                 0.000000
9     1010         Toys                 0.000000
